<a href="https://colab.research.google.com/github/surajsingh2182-bot/low-code-fraud-engine/blob/main/1_Fraud_Data_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# Set a random seed for reproducible results
np.random.seed(42)
n_samples = 100000

# Generate 100,000 realistic financial transaction records
data = {
    'transaction_id': [f"TX{i:06d}" for i in range(n_samples)],
    'amount': np.round(np.random.exponential(scale=150, size=n_samples) + 1, 2),
    'oldbalanceOrg': np.round(np.random.uniform(10, 50000, size=n_samples), 2),
    'type': np.random.choice(['TRANSFER', 'CASH_OUT', 'PAYMENT', 'DEBIT'], size=n_samples, p=[0.3, 0.4, 0.2, 0.1]),
    # Inject a strict 0.2% fraud rate (Class Imbalance)
    'isFraud': np.random.choice([0, 1], size=n_samples, p=[0.998, 0.002])
}

df = pd.DataFrame(data)

# Force transactions with massive amounts and low balances to look more like fraud patterns
df.loc[(df['amount'] > 5000) & (df['oldbalanceOrg'] < 2000), 'isFraud'] = np.random.choice([0, 1], size=len(df[(df['amount'] > 5000) & (df['oldbalanceOrg'] < 2000)]), p=[0.2, 0.8])

# Display the data profile
print("--- FIRST 5 TRANSACTIONS IN LEDGER ---")
display(df.head())

print("\n--- TRANSACTION CLASS DISTRIBUTION ---")
counts = df['isFraud'].value_counts()
print(counts)

fraud_pct = (df['isFraud'].sum() / len(df)) * 100
print(f"\nExact Fraud Percentage on Platform: {fraud_pct:.3f}%")

--- FIRST 5 TRANSACTIONS IN LEDGER ---


,transaction_id,amount,oldbalanceOrg,type,isFraud
0,TX000000,71.39,29043.14,TRANSFER,0
1,TX000001,452.52,26353.31,CASH_OUT,0
2,TX000002,198.51,17558.34,TRANSFER,0
3,TX000003,137.94,24665.70,CASH_OUT,0
4,TX000004,26.44,18261.18,TRANSFER,0



--- TRANSACTION CLASS DISTRIBUTION ---
isFraud
0    99815
1      185
Name: count, dtype: int64

Exact Fraud Percentage on Platform: 0.185%


In [ ]:
import duckdb

# Connect to our in-memory SQL database engine
conn = duckdb.connect()

# Register our existing dataset ('df') as a virtual SQL table named 'transactions'
conn.register('transactions', df)

# Execute BigQuery-compliant SQL to split data into Training (80%) and Testing (20%)
sql_query = """
SELECT
    transaction_id,
    amount,
    oldbalanceOrg,
    type,
    isFraud,
    CASE
        WHEN abs(hash(transaction_id)) % 10 < 8 THEN 'TRAIN'
        ELSE 'TEST'
    END as data_split
FROM transactions
"""

# Run the query and save it into a clean analytical view
analytics_view = conn.execute(sql_query).df()

print("--- CLOUD SQL DATA SPLIT PREVIEW ---")
display(analytics_view.head(10))

print("\n--- ROW COUNT PER DATA SPLIT ---")
print(analytics_view['data_split'].value_counts())

--- CLOUD SQL DATA SPLIT PREVIEW ---


,transaction_id,amount,oldbalanceOrg,type,isFraud,data_split
0,TX000000,71.39,29043.14,TRANSFER,0,TRAIN
1,TX000001,452.52,26353.31,CASH_OUT,0,TRAIN
2,TX000002,198.51,17558.34,TRANSFER,0,TRAIN
3,TX000003,137.94,24665.70,CASH_OUT,0,TEST
4,TX000004,26.44,18261.18,TRANSFER,0,TRAIN
5,TX000005,26.44,32899.99,TRANSFER,0,TRAIN
6,TX000006,9.98,41753.13,CASH_OUT,0,TEST
7,TX000007,302.68,39346.85,PAYMENT,0,TRAIN
8,TX000008,138.86,17041.21,CASH_OUT,0,TRAIN
9,TX000009,185.69,25475.88,TRANSFER,0,TRAIN



--- ROW COUNT PER DATA SPLIT ---
data_split
TRAIN    79997
TEST     20003
Name: count, dtype: int64


In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np

# 1. Prepare our features (X) and our target label (y) from our SQL view
train_data = analytics_view[analytics_view['data_split'] == 'TRAIN']
test_data = analytics_view[analytics_view['data_split'] == 'TEST']

# We will train on transaction 'amount' and 'oldbalanceOrg'
X_train = train_data[['amount', 'oldbalanceOrg']]
y_train = train_data['isFraud']

X_test = test_data[['amount', 'oldbalanceOrg']]
y_test = test_data['isFraud']

# 2. Initialize and Train the Cloud-Native Model (Equivalent to BQML CREATE MODEL)
model = LogisticRegression(class_weight='balanced') # Balanced weights handle our class imbalance!
model.fit(X_train, y_train)

# 3. Predict the continuous fraud probability for our test transactions
test_predictions_prob = model.predict_proba(X_test)[:, 1]

# Add results back to our test table for PM inspection
test_summary = test_data.copy()
test_summary['Fraud_Probability_Score'] = np.round(test_predictions_prob, 4)

print("--- PRODUCTION ENGINE INFERENCE PREVIEW ---")
display(test_summary[['transaction_id', 'amount', 'oldbalanceOrg', 'isFraud', 'Fraud_Probability_Score']].head(10))

--- PRODUCTION ENGINE INFERENCE PREVIEW ---


,transaction_id,amount,oldbalanceOrg,isFraud,Fraud_Probability_Score
3,TX000003,137.94,24665.70,0,0.4990
6,TX000006,9.98,41753.13,0,0.5293
10,TX000010,4.12,542.71,0,0.4720
13,TX000013,36.80,22060.16,0,0.5005
35,TX000035,248.85,4535.71,0,0.4654
37,TX000037,16.42,45368.96,0,0.5341
41,TX000041,103.53,18441.23,0,0.4921
43,TX000043,361.06,38541.62,0,0.5073
44,TX000044,45.92,11028.78,0,0.4846
53,TX000053,338.82,42372.10,0,0.5138


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

# Apply our chosen conservative PM threshold (0.2)
custom_threshold = 0.2
test_summary['Final_Prediction'] = (test_summary['Fraud_Probability_Score'] >= custom_threshold).astype(int)

# Generate the raw Confusion Matrix counts
tn, fp, fn, tp = confusion_matrix(test_summary['isFraud'], test_summary['Final_Prediction']).ravel()

print("=== COCHING EVALUATION DASHBOARD ===")
print(f"True Negatives (Legit allowed through): {tn}")
print(f"False Positives (False Alarms triggered): {fp}")
print(f"False Negatives (SLIPPED FRAUD!): {fn}")
print(f"True Positives (Fraud successfully caught): {tp}")

# Calculate exact Recall for the PM dashboard
recall = tp / (tp + fn)
print(f"\nFinal Engine Recall: {recall*100:.2f}%")

=== COCHING EVALUATION DASHBOARD ===
True Negatives (Legit allowed through): 0
False Positives (False Alarms triggered): 19967
False Negatives (SLIPPED FRAUD!): 0
True Positives (Fraud successfully caught): 36

Final Engine Recall: 100.00%


In [ ]:
import json

def real_time_fraud_gateway(api_payload_json):
    \"\"\"
    Simulates a live production API endpoint receiving data from a front-end app.
    \"\"\"
    # 1. Parse the incoming transaction data from the app
    payload = json.loads(api_payload_json)

    # 2. Extract the features the model needs
    amount = payload['amount']
    old_balance = payload['oldbalanceOrg']

    # 3. Format data for the model and run real-time inference
    input_data = [[amount, old_balance]]
    probability = model.predict_proba(input_data)[0][1]

    # 4. Apply our PM automation routing rules
    if probability < 0.20:
        action = "APPROVE"
        status_code = 200
    elif 0.20 <= probability < 0.80:
        action = "TRIGGER_LOW_CODE_SMS_VERIFICATION"
        status_code = 202
    else:
        action = "IMMEDIATE_LOCK_AND_BLOCK"
        status_code = 403

    # Return the response payload back to the calling application
    response = {
        "transaction_id": payload.get("transaction_id", "UNKNOWN"),
        "fraud_probability": round(float(probability), 4),
        "recommended_action": action,
        "status_code": status_code
    }
    return json.dumps(response, indent=4)

# --- SIMULATE A LIVE TRANSACTIONS COMING FROM THE WEB ---
print("--- SCENARIO 1: A small routine payment ---")
sample_payload_1 = '{"transaction_id": "TX999991", "amount": 15.50, "oldbalanceOrg": 500.00}'
print(real_time_fraud_gateway(sample_payload_1))

print("\n--- SCENARIO 2: A highly suspicious massive withdrawal ---")
sample_payload_2 = '{"transaction_id": "TX999992", "amount": 8500.00, "oldbalanceOrg": 150.00}'
print(real_time_fraud_gateway(sample_payload_2))

SyntaxError: unexpected character after line continuation character (870762792.py, line 4)

In [ ]:
import json

def real_time_fraud_gateway(api_payload_json):
    # 1. Parse the incoming transaction data from the front-end app
    payload = json.loads(api_payload_json)

    # 2. Extract the features the model needs
    amount = payload['amount']
    old_balance = payload['oldbalanceOrg']

    # 3. Format data for the model and run real-time inference
    input_data = [[amount, old_balance]]
    probability = model.predict_proba(input_data)[0][1]

    # 4. Apply our PM automation routing rules
    if probability < 0.20:
        action = "APPROVE"
        status_code = 200
    elif 0.20 <= probability < 0.80:
        action = "TRIGGER_LOW_CODE_SMS_VERIFICATION"
        status_code = 202
    else:
        action = "IMMEDIATE_LOCK_AND_BLOCK"
        status_code = 403

    # Return the response payload back to the calling application
    response = {
        "transaction_id": payload.get("transaction_id", "UNKNOWN"),
        "fraud_probability": round(float(probability), 4),
        "recommended_action": action,
        "status_code": status_code
    }
    return json.dumps(response, indent=4)

# --- SIMULATE LIVE TRANSACTIONS COMING FROM THE WEB ---
print("--- SCENARIO 1: A small routine payment ---")
sample_payload_1 = '{"transaction_id": "TX999991", "amount": 15.50, "oldbalanceOrg": 500.00}'
print(real_time_fraud_gateway(sample_payload_1))

print("\n--- SCENARIO 2: A highly suspicious massive withdrawal ---")
sample_payload_2 = '{"transaction_id": "TX999992", "amount": 8500.00, "oldbalanceOrg": 150.00}'
print(real_time_fraud_gateway(sample_payload_2))

--- SCENARIO 1: A small routine payment ---
{
    "transaction_id": "TX999991",
    "fraud_probability": 0.4714,
    "recommended_action": "TRIGGER_LOW_CODE_SMS_VERIFICATION",
    "status_code": 202
}

--- SCENARIO 2: A highly suspicious massive withdrawal ---
{
    "transaction_id": "TX999992",
    "fraud_probability": 0.1399,
    "recommended_action": "APPROVE",
    "status_code": 200
}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [1]:
import json

def real_time_fraud_gateway(api_payload_json):
    """
    Simulates a live production API endpoint receiving data from a front-end app.
    """
    # 1. Parse the incoming transaction data from the app
    payload = json.loads(api_payload_json)

    # 2. Extract the features the model needs
    amount = payload['amount']
    old_balance = payload['oldbalanceOrg']

    # 3. Format data for the model and run real-time inference
    input_data = [[amount, old_balance]]
    probability = model.predict_proba(input_data)[0][1]

    # 4. Apply our PM automation routing rules
    if probability < 0.20:
        action = "APPROVE"
        status_code = 200
    elif 0.20 <= probability < 0.80:
        action = "TRIGGER_LOW_CODE_SMS_VERIFICATION"
        status_code = 202
    else:
        action = "IMMEDIATE_LOCK_AND_BLOCK"
        status_code = 403

    # Return the response payload back to the calling application
    response = {
        "transaction_id": payload.get("transaction_id", "UNKNOWN"),
        "fraud_probability": round(float(probability), 4),
        "recommended_action": action,
        "status_code": status_code
    }
    return json.dumps(response, indent=4)

# --- SIMULATE A LIVE TRANSACTIONS COMING FROM THE WEB ---
print("--- SCENARIO 1: A small routine payment ---")
sample_payload_1 = '{"transaction_id": "TX999991", "amount": 15.50, "oldbalanceOrg": 500.00}'
print(real_time_fraud_gateway(sample_payload_1))

print("\n--- SCENARIO 2: A highly suspicious massive withdrawal ---")
sample_payload_2 = '{"transaction_id": "TX999992", "amount": 8500.00, "oldbalanceOrg": 150.00}'
print(real_time_fraud_gateway(sample_payload_2))

SyntaxError: unexpected character after line continuation character (870762792.py, line 4)

In [2]:
import json

def real_time_fraud_gateway(api_payload_json):
    # 1. Parse the incoming transaction data from the front-end app
    payload = json.loads(api_payload_json)

    # 2. Extract the features the model needs
    amount = payload['amount']
    old_balance = payload['oldbalanceOrg']

    # 3. Format data for the model and run real-time inference
    input_data = [[amount, old_balance]]
    probability = model.predict_proba(input_data)[0][1]

    # 4. Apply our PM automation routing rules
    if probability < 0.20:
        action = "APPROVE"
        status_code = 200
    elif 0.20 <= probability < 0.80:
        action = "TRIGGER_LOW_CODE_SMS_VERIFICATION"
        status_code = 202
    else:
        action = "IMMEDIATE_LOCK_AND_BLOCK"
        status_code = 403

    # Return the response payload back to the calling application
    response = {
        "transaction_id": payload.get("transaction_id", "UNKNOWN"),
        "fraud_probability": round(float(probability), 4),
        "recommended_action": action,
        "status_code": status_code
    }
    return json.dumps(response, indent=4)

# --- SIMULATE LIVE TRANSACTIONS COMING FROM THE WEB ---
print("--- SCENARIO 1: A small routine payment ---")
sample_payload_1 = '{"transaction_id": "TX999991", "amount": 15.50, "oldbalanceOrg": 500.00}'
print(real_time_fraud_gateway(sample_payload_1))

print("\n--- SCENARIO 2: A highly suspicious massive withdrawal ---")
sample_payload_2 = '{"transaction_id": "TX999992", "amount": 8500.00, "oldbalanceOrg": 150.00}'
print(real_time_fraud_gateway(sample_payload_2))

--- SCENARIO 1: A small routine payment ---


NameError: name 'model' is not defined

In [3]:
import pandas as pd
import numpy as np
import json
from sklearn.linear_model import LogisticRegression

# 1. EMULATE TRAINED MODEL IN MEMORY
# (Creating a quick baseline dataset to initialize the model object instantly)
np.random.seed(42)
mock_samples = 1000
mock_data = {
    'amount': np.random.exponential(scale=150, size=mock_samples),
    'oldbalanceOrg': np.random.uniform(10, 50000, size=mock_samples),
    'isFraud': np.random.choice([0, 1], size=mock_samples, p=[0.99, 0.01])
}
mock_df = pd.DataFrame(mock_data)

# Force fraud patterns for the model to learn
mock_df.loc[(mock_df['amount'] > 5000) & (mock_df['oldbalanceOrg'] < 2000), 'isFraud'] = 1

# Train and define 'model' inside this specific runtime block
model = LogisticRegression(class_weight='balanced')
model.fit(mock_df[['amount', 'oldbalanceOrg']], mock_df['isFraud'])


# 2. THE LIVE API GATEWAY FUNCTION
def real_time_fraud_gateway(api_payload_json):
    # Parse the incoming transaction data from the front-end app
    payload = json.loads(api_payload_json)

    # Extract the features the model needs
    amount = payload['amount']
    old_balance = payload['oldbalanceOrg']

    # Format data for the model and run real-time inference
    input_data = [[amount, old_balance]]
    probability = model.predict_proba(input_data)[0][1]

    # Apply our PM automation routing rules
    if probability < 0.20:
        action = "APPROVE"
        status_code = 200
    elif 0.20 <= probability < 0.80:
        action = "TRIGGER_LOW_CODE_SMS_VERIFICATION"
        status_code = 202
    else:
        action = "IMMEDIATE_LOCK_AND_BLOCK"
        status_code = 403

    # Return the response payload back to the calling application
    response = {
        "transaction_id": payload.get("transaction_id", "UNKNOWN"),
        "fraud_probability": round(float(probability), 4),
        "recommended_action": action,
        "status_code": status_code
    }
    return json.dumps(response, indent=4)


# --- SIMULATE LIVE TRANSACTIONS COMING FROM THE WEB FRONT-END ---
print("--- SCENARIO 1: A small routine payment ---")
sample_payload_1 = '{"transaction_id": "TX999991", "amount": 15.50, "oldbalanceOrg": 500.00}'
print(real_time_fraud_gateway(sample_payload_1))

print("\n--- SCENARIO 2: A highly suspicious massive withdrawal ---")
sample_payload_2 = '{"transaction_id": "TX999992", "amount": 8500.00, "oldbalanceOrg": 150.00}'
print(real_time_fraud_gateway(sample_payload_2))

--- SCENARIO 1: A small routine payment ---
{
    "transaction_id": "TX999991",
    "fraud_probability": 0.396,
    "recommended_action": "TRIGGER_LOW_CODE_SMS_VERIFICATION",
    "status_code": 202
}

--- SCENARIO 2: A highly suspicious massive withdrawal ---
{
    "transaction_id": "TX999992",
    "fraud_probability": 1.0,
    "recommended_action": "IMMEDIATE_LOCK_AND_BLOCK",
    "status_code": 403
}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
